## 1. Setup and Imports

In [ ]:
import sys
import os

# Add src to path
sys.path.append('../src')

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
import pandas as pd

from data_loader import BrainTumorDataLoader
from preprocessing import DataPreprocessor
from model import BrainTumorCNN

%matplotlib inline
plt.style.use('seaborn-v0_8')
sns.set_palette('husl')

## 2. Data Loading

In [ ]:
# Initialize data loader
data_loader = BrainTumorDataLoader(img_size=(150, 150))

# Paths to data (adjust as needed)
train_path = '../../Dataset/Training'
test_path = '../../Dataset/Testing'

# Load data
print("Loading training data...")
train_data, train_labels = data_loader.load_train_data(train_path)

print("\nLoading testing data...")
test_data, test_labels = data_loader.load_test_data(test_path)

## 3. Data Exploration

In [ ]:
# Dataset statistics
print("Dataset Statistics:")
print(f"Training samples: {train_data.shape[0]}")
print(f"Testing samples: {test_data.shape[0]}")
print(f"Image shape: {train_data.shape[1:]}")
print(f"Number of classes: {train_labels.shape[1]}")

In [ ]:
# Class distribution
train_class_counts = np.argmax(train_labels, axis=1)
test_class_counts = np.argmax(test_labels, axis=1)

class_names = data_loader.classes

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Training distribution
axes[0].bar(class_names, np.bincount(train_class_counts))
axes[0].set_title('Training Set - Class Distribution')
axes[0].set_xlabel('Class')
axes[0].set_ylabel('Count')
axes[0].tick_params(axis='x', rotation=45)

# Testing distribution
axes[1].bar(class_names, np.bincount(test_class_counts))
axes[1].set_title('Testing Set - Class Distribution')
axes[1].set_xlabel('Class')
axes[1].set_ylabel('Count')
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

## 4. Visualize Sample Images

In [ ]:
# Display sample images from each class
fig, axes = plt.subplots(4, 4, figsize=(12, 12))

for class_idx in range(4):
    # Get indices for this class
    class_indices = np.where(train_class_counts == class_idx)[0]
    
    # Select 4 random samples
    sample_indices = np.random.choice(class_indices, 4, replace=False)
    
    for i, idx in enumerate(sample_indices):
        axes[class_idx, i].imshow(train_data[idx])
        axes[class_idx, i].axis('off')
        if i == 0:
            axes[class_idx, i].set_ylabel(class_names[class_idx], fontsize=12)

plt.suptitle('Sample Images from Each Class', fontsize=16, y=0.995)
plt.tight_layout()
plt.show()

## 5. Pixel Intensity Analysis

In [ ]:
# Analyze pixel intensity distributions
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for i, channel in enumerate(['Red', 'Green', 'Blue']):
    axes[i].hist(train_data[:, :, :, i].flatten(), bins=50, alpha=0.7)
    axes[i].set_title(f'{channel} Channel Distribution')
    axes[i].set_xlabel('Pixel Value')
    axes[i].set_ylabel('Frequency')

plt.tight_layout()
plt.show()

## 6. Model Architecture

In [ ]:
# Build and visualize model
cnn = BrainTumorCNN(input_shape=(150, 150, 3), num_classes=4)
model = cnn.build_model()
cnn.compile_model()

# Display model summary
model.summary()

## 7. Data Augmentation Preview

In [ ]:
# Visualize augmented images
preprocessor = DataPreprocessor()
preprocessor.create_augmentation_generator(
    rotation_range=15,
    zoom_range=0.1,
    width_shift_range=0.1,
    height_shift_range=0.1,
    horizontal_flip=True
)

# Take one sample image
sample_img = train_data[0:1]
sample_label = train_labels[0:1]

# Generate augmented versions
datagen = preprocessor.get_generator()
datagen.fit(sample_img)

fig, axes = plt.subplots(2, 4, figsize=(12, 6))
axes = axes.flatten()

# Original image
axes[0].imshow(sample_img[0].astype('uint8'))
axes[0].set_title('Original')
axes[0].axis('off')

# Augmented images
i = 1
for batch in datagen.flow(sample_img, sample_label, batch_size=1):
    axes[i].imshow(batch[0][0].astype('uint8'))
    axes[i].set_title(f'Augmented {i}')
    axes[i].axis('off')
    i += 1
    if i >= 8:
        break

plt.suptitle('Data Augmentation Examples', fontsize=16)
plt.tight_layout()
plt.show()

## 8. Class-wise Image Statistics

In [ ]:
# Calculate mean and std for each class
class_stats = []

for class_idx in range(4):
    class_images = train_data[train_class_counts == class_idx]
    
    stats = {
        'Class': class_names[class_idx],
        'Mean': class_images.mean(),
        'Std': class_images.std(),
        'Min': class_images.min(),
        'Max': class_images.max()
    }
    class_stats.append(stats)

stats_df = pd.DataFrame(class_stats)
print("\nClass-wise Image Statistics:")
print(stats_df)

## 9. Next Steps

Based on the exploration:
1. The dataset is relatively balanced
2. Images have good contrast and quality
3. Data augmentation will help with generalization
4. Model architecture is appropriate for the task

Ready to proceed with training!